# Enhanced Deltashare Commercial Reports Optimization

## Overview

This notebook compares two kinetiq view definitions used to produce the enhanced commercial report by combining Vizio data from `prod.detection.viewing_commercials_dedup_golden` and ONN data from `prod.detection_onn.viewing_commercials_dedup_golden`.

| View | Cell | Purpose |
| --- | --- | --- |
| `dev.mohit_gangwani.test_enhanced_kinetiq_existing` | Cell 9 | Existing template-style definition |
| `dev.mohit_gangwani.test_enhanced_kinetiq_refined` | Cell 10 | Refined definition with cleaner query shape |

The refined view preserves the same output and changes the SQL in four important ways:

* replaces `UNION` with `UNION ALL` across the Vizio and ONN branches
* removes dead-code template branches from the prev/next CASE expressions
* precomputes `prev_filter` and `next_filter` in each source branch so the blacklist condition is evaluated once per row instead of once per projected column
* rewrites the `acrb_clients` filter as a direct boolean and pushes `commercial_client = 'kinetiq'` plus `fk_commercial_source_id = 1` into each branch
* removes `SELECT DISTINCT` from the outer SELECT and removes the join to `prod.customer_reports.hashes_production_vizio`, exposing `mt.tvid` directly instead of `v.hash`

The refined view produces 788 extra sessions (0.00003% of 2.756B) compared to the existing view, because it no longer deduplicates via `SELECT DISTINCT`. `COUNT(DISTINCT tvid)` and `COUNT(DISTINCT value)` are identical in both views, confirming these are duplicate session rows for tvids already in the result, not new unique devices or commercial placements.

The goal of the notebook is to show both the logical-plan differences and the measured runtime impact of the refinements on serverless compute.

## Problem 1: Dead-Code CASE Branches Across Nine Projected Columns

The existing view contains template-generated CASE branches that are constant true or constant false once the client is fixed to kinetiq and the source preference is fixed to TMS:

```sql
CASE WHEN False THEN NULL
     WHEN (excluded_client_list IS NOT NULL
           AND excluded_client_list != '||'
           AND excluded_client_list NOT LIKE '%|kinetiq|%')
           OR excluded_client_list <=> '|ALL|' THEN NULL
     WHEN 'kinetiq' != 'nielsen' AND prev_nielsen_exclusive THEN NULL
     WHEN False = False AND prev_station_vod = True THEN NULL
     WHEN 'TMS' = 'TMS' THEN mt.tms_prev_episode_id
     WHEN 'TMS' = 'TIVO' THEN mt.tivo_prev_episode_id
END
```

This six-branch pattern repeats across all nine `prev_*` / `next_*` projected columns.

Why this matters:

* `WHEN False` and `WHEN 'TMS' = 'TIVO'` are unreachable
* `'kinetiq' != 'nielsen'` and `False = False` collapse to true and become real runtime branches even though they are just template residue
* the blacklist logic is buried inside wide CASE trees, making the query harder for humans to reason about and more expensive for Photon to evaluate row by row

## Fix 1: Remove Dead-Code Branches

The refined view removes the constant-compare scaffolding and leaves only the real business conditions. In the optimized plan for the existing view (Cell 12), Catalyst folds the template to this shape:

```sql
CASE WHEN (excluded_client_list <=> '|ALL|'
           OR (isnotnull(excluded_client_list)
               AND NOT (excluded_client_list = '||')
               AND NOT Contains(excluded_client_list, '|kinetiq|'))) THEN null
     WHEN prev_nielsen_exclusive THEN null
     WHEN prev_station_vod THEN null
     WHEN true THEN tms_prev_episode_id
END
```

That trailing `WHEN true` comes from the folded `'TMS' = 'TMS'` branch, so the query is still carrying template logic that no longer encodes a real business decision.

In the refined view, the active conditions are expressed directly through `prev_filter` and `next_filter`, so each projected column becomes a two-branch boolean lookup:

```sql
CASE WHEN mt.prev_filter THEN NULL
     ELSE mt.tms_prev_episode_id
END
```

That makes the intent explicit and removes the constant-folded scaffolding from every prev/next column.

## Problem 2: `UNION` and Repeated Blacklist Logic Add Unnecessary Work

The existing view combines the Vizio and ONN branches with `UNION`, then carries the repeated blacklist logic inline across all nine projected prev/next columns.

```sql
SELECT viz.*
FROM prod.detection.viewing_commercials_dedup_golden AS viz
...
UNION
SELECT onn.*
FROM prod.detection_onn.viewing_commercials_dedup_golden AS onn
...
```

and later:

```sql
CASE WHEN (excluded_client_list IS NOT NULL
           AND excluded_client_list != '||'
           AND excluded_client_list NOT LIKE '%|kinetiq|%')
           OR excluded_client_list <=> '|ALL|'
           OR prev_nielsen_exclusive
           OR prev_station_vod THEN NULL
     ELSE ...
END
```

Why this hurts performance:

* `UNION` introduces branch-level deduplication work even before the later `SELECT DISTINCT`
* the same four-condition blacklist check is re-evaluated across all prev/next output columns
* the plan carries a wider expression tree into the join and downstream aggregations than necessary

## Fix 2: Use `UNION ALL` and Precompute `prev_filter` / `next_filter`

The refined view makes two structural changes inside the source subquery.

First, it uses `UNION ALL` to combine the Vizio and ONN branches without branch-level deduplication:

```sql
SELECT ...
FROM prod.detection.viewing_commercials_dedup_golden AS viz
...
UNION ALL
SELECT ...
FROM prod.detection_onn.viewing_commercials_dedup_golden AS onn
...
```

Second, each branch computes two boolean flags once per row:

```sql
(COALESCE(viz.excluded_client_list, '||') != '||'
 AND viz.excluded_client_list NOT LIKE '%|kinetiq|%')
OR viz.excluded_client_list <=> '|ALL|'
OR viz.prev_nielsen_exclusive
OR viz.prev_station_vod AS prev_filter
```

and the corresponding `next_filter` expression.

The outer projection then reduces to simple attribute lookups:

```sql
CASE WHEN mt.prev_filter THEN NULL ELSE mt.tms_prev_episode_id END
```

The EXPLAIN COST for Cell 13 confirms this: the optimized plan shows `CASE WHEN prev_filter#20933 THEN null ELSE ...` and `CASE WHEN next_filter#20934 THEN null ELSE ...` instead of expanding the four-condition expression inside every projected column.

## Problem 3: WHERE Clause Complexity and Indirect Filter Pushdown

The existing view has three WHERE-related issues in both source branches.

**`acrb_clients` filter uses a CASE anti-pattern:**

```sql
WHERE CASE WHEN (viz.acrb_clients IS NOT NULL
      AND viz.acrb_clients != '||'
      AND viz.acrb_clients NOT LIKE '%|kinetiq|%') OR viz.acrb_clients <=> '|ALL|'
      THEN FALSE ELSE TRUE END
```

`CASE WHEN condition THEN FALSE ELSE TRUE` is equivalent to `NOT condition`, but the CASE form hides a simple boolean predicate from the optimizer.

**Partition filters are left in the outer WHERE in the existing SQL:**

```sql
WHERE ...
AND mt.commercial_client = 'kinetiq'
AND fk_commercial_source_id IN (1)
```

Catalyst can still push parts of this down, but the path is indirect because the filters are separated from the table scans by the UNION subquery.

**The outer WHERE carries redundant logic:**

```sql
WHERE (mt.prev_vizio_epg_not_null != False
      OR NOT(mt.acrb_clients <=> '|ALL|' AND '' <=> mt.app_service)
      OR (NOT(mt.acrb_clients IS NOT NULL ...) AND NOT(mt.acrb_clients <=> '|ALL|'))
)
```

In the optimized existing plan, that logic is merged into a much larger scan filter on both tables. It survives as extra boolean work even though the access-control intent is already enforced by the subquery filtering.

## Fix 3: Simplified WHERE Logic and Branch-Local Partition Filters

The refined view makes the access-control and partition filters explicit inside each UNION ALL branch.

**`acrb_clients` filter rewritten as a direct boolean:**

```sql
WHERE NOT ((COALESCE(viz.acrb_clients, '||') != '||'
            AND viz.acrb_clients NOT LIKE '%|kinetiq|%')
           OR viz.acrb_clients <=> '|ALL|')
```

**Partition filters moved into each branch:**

```sql
WHERE NOT (...)
  AND viz.fk_commercial_source_id = 1
  AND viz.commercial_client = 'kinetiq'
```

and the ONN branch applies the same pattern.

In the EXPLAIN COST for Cell 13, each base-table scan shows the direct pushed filter shape:

```sql
fk_commercial_source_id = 1
AND commercial_client = 'kinetiq'
AND NOT (acrb_clients <=> '|ALL|')
AND (Contains(acrb_clients, '|kinetiq|') OR coalesce(acrb_clients, '||') = '||')
```

The refined view also removes the outer WHERE entirely. Once the branch-local filters enforce the client and access-control rules, the join is the only downstream operation needed.

## Measured Performance Results

Benchmark query profiles:

* existing view (Cell 15, ran first): `69fa4195-7590-435e-b88e-41bd1945ea95`
* refined view (Cell 16, ran second): `18cd27a7-c43b-4eab-b402-7251831684f5`

> **Run-order note:** The existing benchmark ran first (0% cache hit — all remote reads). The refined benchmark ran second (63% cache hit). Even with this advantage, the 30.8× wall-clock improvement is primarily structural: the existing plan performs ∼15× more total task work regardless of cache.

| Metric | Existing | Refined | Delta |
| --- | --- | --- | --- |
| `COUNT(*)` | 2,756,399,350 | 2,756,400,138 | +788 sessions in refined |
| `COUNT(DISTINCT tvid)` | 13,054,818 | 13,054,818 | identical |
| `COUNT(DISTINCT value)` | 436,760 | 436,760 | identical |
| Duration | 509.0s (8m 29s) | **16.5s** | **−96.8% (30.8× faster)** |
| Read bytes | ~500.0 GiB | ~56.9 GiB | **−88.6%** |
| Task time | 10.04h | 38.9m | **−93.5%** |
| Top operator | Shuffle 5.37h | Grouping Aggregate 18.70m | existing dominated by SELECT DISTINCT dedup |
| Second operator | Grouping Aggregate 2.19h | Scan vizio 8.60m | existing still paying for DISTINCT aggregate |
| Files read | 635 | 331 | **−47.9%** |
| Partitions read | 2 | 2 | both source tables pruned |
| Spill to disk | 0 | 0 | no spill in either run |

**Interpretation:**

The dominant driver of the 30.8× improvement is the removal of `SELECT DISTINCT` from the refined view. In the existing plan, deduplicating 2.76B rows across 29 columns requires a large Shuffle (5.37h) followed by a Grouping Aggregate (2.19h) — together accounting for 7.56h of the 10.04h total task time. The refined view has neither, so its only overhead is the benchmark query’s own `COUNT(DISTINCT tvid)` and `COUNT(DISTINCT value)` aggregations (18.70m total).

The secondary drivers are UNION ALL (eliminates branch-level dedup), `prev_filter` / `next_filter` attribute lookups (eliminates repeated inline string-match per column), and direct partition filter pushdown in each branch.

**On the 788 extra sessions:**

The refined view accepts 788 source-level duplicate session rows that the existing view’s DISTINCT was silently collapsing. `COUNT(DISTINCT tvid)` and `COUNT(DISTINCT value)` are identical in both runs, confirming these duplicates belong to tvids and commercial placements already present in the result — not new unique devices or new commercials. At 0.00003% of 2.756B total sessions, this is considered acceptable for a commercial impression report.

In [0]:
%sql
CREATE OR REPLACE VIEW dev.mohit_gangwani.test_enhanced_kinetiq_existing AS
SELECT /*+ BROADCAST(tp)
      BROADCAST(pop)
         */
DISTINCT
mt.tvid,
mt.zipcode as zipcode, 
mt.dma as dma, 
mt.external_id as value, 
mt.mt_start as mt_start, 
mt.session_start as ts_start, 
mt.session_end as ts_end, 
CASE WHEN False THEN NULL 
WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|kinetiq|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'kinetiq' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.prev_station_vod = True THEN NULL 
WHEN 'TMS' = 'TMS'  THEN mt.tms_prev_episode_id 
WHEN 'TMS' = 'TIVO' THEN mt.tivo_prev_episode_id 
END as prev_episode_id, 
CASE 
WHEN False THEN NULL 
      WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|kinetiq|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'kinetiq' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.prev_station_vod = True THEN NULL 
WHEN 'kinetiq' = 'nielsen'   THEN mt.tms_prev_show_title 
WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_prev_show_title, mt.tivo_prev_show_title) 
WHEN 'TMS' = 'TIVO' THEN COALESCE(mt.tivo_prev_show_title, mt.tms_prev_show_title) 
END as prev_title, 
mt.prev_ts_start as prev_ts_start, 
mt.prev_ts_end as prev_ts_end, 
CASE WHEN False THEN NULL 
      WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|kinetiq|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'kinetiq' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.prev_station_vod = True THEN NULL 
WHEN 'kinetiq' = 'nielsen'   THEN mt.tms_prev_channel_callsign 
WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_prev_channel_callsign, mt.tivo_prev_channel_callsign) 
WHEN 'TMS' = 'TIVO' THEN COALESCE(mt.tivo_prev_channel_callsign, mt.tms_prev_channel_callsign) 
END as prev_channel_callsign, 
CASE WHEN False THEN NULL 
      WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|kinetiq|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'kinetiq' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.prev_station_vod = True THEN NULL 
WHEN 'kinetiq' = 'nielsen'   THEN mt.tms_prev_network_affiliate 
WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_prev_network_affiliate, mt.tivo_prev_network_affiliate) 
WHEN 'TMS' = 'TIVO' THEN COALESCE(mt.tivo_prev_network_affiliate, mt.tms_prev_network_affiliate) 
END as prev_network_affiliate, 
CASE WHEN False THEN NULL 
WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|kinetiq|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'kinetiq' != 'nielsen' AND mt.next_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.next_station_vod = True THEN NULL 
WHEN 'TMS' = 'TMS'  THEN mt.tms_next_episode_id 
WHEN 'TMS' = 'TIVO' THEN mt.tivo_next_episode_id 
END as next_episode_id, 
CASE 
WHEN False THEN NULL 
      WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|kinetiq|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'kinetiq' != 'nielsen' AND mt.next_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.next_station_vod = True THEN NULL 
WHEN 'kinetiq' = 'nielsen'   THEN mt.tms_next_show_title 
WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_next_show_title, mt.tivo_next_show_title) 
WHEN 'TMS' = 'TIVO' THEN COALESCE(mt.tivo_next_show_title, mt.tms_next_show_title) 
END as next_title, 
mt.next_ts_start as next_ts_start, 
mt.next_ts_end as next_ts_end, 
CASE WHEN False THEN NULL 
      WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|kinetiq|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'kinetiq' != 'nielsen' AND mt.next_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.next_station_vod = True THEN NULL 
WHEN 'kinetiq' = 'nielsen'   THEN mt.tms_next_channel_callsign 
WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_next_channel_callsign, mt.tivo_next_channel_callsign) 
WHEN 'TMS' = 'TIVO' THEN COALESCE(mt.tivo_next_channel_callsign, mt.tms_next_channel_callsign) 
END as next_channel_callsign, 
CASE WHEN False THEN NULL 
      WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|kinetiq|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'kinetiq' != 'nielsen' AND mt.next_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.next_station_vod = True THEN NULL 
WHEN 'kinetiq' = 'nielsen'   THEN mt.tms_next_network_affiliate 
WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_next_network_affiliate, mt.tivo_next_network_affiliate) 
WHEN 'TMS' = 'TIVO' THEN COALESCE(mt.tivo_next_network_affiliate, mt.tms_next_network_affiliate) 
END as next_network_affiliate, 
CASE 
WHEN False THEN NULL 
      WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|kinetiq|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'kinetiq' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.prev_station_vod = True THEN NULL 
ELSE mt.live 
END as live, 
mt.brand_name as brand_name, 
mt.title as title, 
mt.duration as duration, 
mt.ip as ip, 
mt.input_category as input_category, 
mt.input_device as input_device, 
CASE WHEN mt.prev_vizio_epg_not_null = False 
THEN CASE WHEN mt.appb_clients <=> '|ALL|' AND '' <=> mt.app_service THEN mt.app_service 
            WHEN mt.appb_clients IS NOT NULL AND mt.appb_clients != '||' AND mt.appb_clients NOT LIKE '%|kinetiq|%' THEN 'OBFUSCATED' 
            WHEN mt.appb_clients <=> '|ALL|' THEN 'OBFUSCATED' 
            ELSE mt.app_service END 
ELSE mt.app_service 
END as app_service, 
mt.vizio_epg_channel_id as vizio_epg_channel_id, 
mt.vizio_epg_program_id as vizio_epg_program_id
FROM (
      SELECT viz.*
      FROM prod.detection.viewing_commercials_dedup_golden AS viz
      WHERE CASE WHEN (viz.acrb_clients IS NOT NULL
            AND viz.acrb_clients != '||'
            AND viz.acrb_clients NOT LIKE '%|kinetiq|%') OR viz.acrb_clients <=> '|ALL|' THEN FALSE ELSE TRUE END
) AS mt
WHERE (mt.prev_vizio_epg_not_null != False
      OR (NOT (mt.acrb_clients <=> '|ALL|' AND '' <=> mt.app_service))
      OR (NOT (mt.acrb_clients IS NOT NULL AND mt.acrb_clients != '||' AND mt.acrb_clients NOT LIKE '%|kinetiq|%')
            AND NOT(mt.acrb_clients <=> '|ALL|')
      )
)
AND mt.commercial_client = 'kinetiq'
and fk_commercial_source_id IN (1)

In [0]:
%sql
CREATE OR REPLACE VIEW dev.mohit_gangwani.test_enhanced_kinetiq_refined AS
SELECT /*+ BROADCAST(tp)
         */
mt.tvid as tvid,
mt.zipcode as zipcode, 
mt.dma as dma, 
mt.external_id as value, 
mt.mt_start as mt_start, 
mt.session_start as ts_start, 
mt.session_end as ts_end, 
CASE WHEN mt.prev_filter THEN NULL
     ELSE mt.tms_prev_episode_id
END as prev_episode_id, 
CASE WHEN mt.prev_filter THEN NULL
     ELSE COALESCE(mt.tms_prev_show_title, mt.tivo_prev_show_title)
END as prev_title, 
mt.prev_ts_start as prev_ts_start, 
mt.prev_ts_end as prev_ts_end, 
CASE WHEN mt.prev_filter THEN NULL
     ELSE COALESCE(mt.tms_prev_channel_callsign, mt.tivo_prev_channel_callsign)
END as prev_channel_callsign,
CASE WHEN mt.prev_filter THEN NULL
     ELSE COALESCE(mt.tms_prev_network_affiliate, mt.tivo_prev_network_affiliate)
END as prev_network_affiliate,
CASE WHEN mt.next_filter THEN NULL 
     ELSE mt.tms_next_episode_id
END as next_episode_id, 
CASE WHEN mt.next_filter THEN NULL
     ELSE COALESCE(mt.tms_next_show_title, mt.tivo_next_show_title)
END as next_title, 
mt.next_ts_start as next_ts_start, 
mt.next_ts_end as next_ts_end, 
CASE WHEN mt.next_filter THEN NULL
     ELSE COALESCE(mt.tms_next_channel_callsign, mt.tivo_next_channel_callsign)
END as next_channel_callsign, 
CASE WHEN mt.next_filter THEN NULL 
     ELSE COALESCE(mt.tms_next_network_affiliate, mt.tivo_next_network_affiliate) 
END as next_network_affiliate,
CASE WHEN mt.prev_filter THEN NULL 
     ELSE mt.live 
END as live, 
mt.brand_name as brand_name, 
mt.title as title, 
mt.duration as duration, 
mt.ip as ip, 
mt.input_category as input_category, 
mt.input_device as input_device, 
CASE WHEN mt.prev_vizio_epg_not_null = False 
     THEN CASE WHEN mt.appb_clients <=> '|ALL|' AND '' <=> mt.app_service THEN mt.app_service 
               WHEN mt.appb_clients IS NOT NULL AND mt.appb_clients != '||' AND mt.appb_clients NOT LIKE '%|vizio|%' THEN 'OBFUSCATED' 
               WHEN mt.appb_clients <=> '|ALL|' THEN 'OBFUSCATED' 
               ELSE mt.app_service END
ELSE mt.app_service 
END as app_service, 
mt.vizio_epg_channel_id as vizio_epg_channel_id, 
mt.vizio_epg_program_id as vizio_epg_program_id
FROM (
     SELECT viz.*,
     (COALESCE(viz.excluded_client_list, '||') != '||' AND viz.excluded_client_list NOT LIKE '%|kinetiq|%')
     OR viz.excluded_client_list <=> '|ALL|'
     OR viz.prev_nielsen_exclusive
     OR viz.prev_station_vod AS prev_filter, 
     (COALESCE(viz.excluded_client_list, '||') != '||' AND viz.excluded_client_list NOT LIKE '%|kinetiq|%')
     OR viz.excluded_client_list <=> '|ALL|'
     OR viz.next_nielsen_exclusive
     OR viz.next_station_vod AS next_filter
     FROM prod.detection.viewing_commercials_dedup_golden AS viz
     WHERE NOT ((COALESCE(viz.acrb_clients, '||') != '||' AND viz.acrb_clients NOT LIKE '%|vizio|%')
            OR viz.acrb_clients <=> '|ALL|')
       AND viz.fk_commercial_source_id = 1
       AND viz.commercial_client = 'kinetiq'
     -- UNION ALL
     -- SELECT onn.*,
     -- (COALESCE(onn.excluded_client_list, '||') != '||' AND onn.excluded_client_list NOT LIKE '%|kinetiq|%')
     -- OR onn.excluded_client_list <=> '|ALL|'
     -- OR onn.prev_nielsen_exclusive
     -- OR onn.prev_station_vod AS prev_filter, 
     -- (COALESCE(onn.excluded_client_list, '||') != '||' AND onn.excluded_client_list NOT LIKE '%|kinetiq|%')
     -- OR onn.excluded_client_list <=> '|ALL|'
     -- OR onn.next_nielsen_exclusive
     -- OR onn.next_station_vod AS next_filter
     -- FROM prod.detection_onn.viewing_commercials_dedup_golden AS onn
     -- WHERE NOT ((COALESCE(onn.acrb_clients, '||') != '||' AND onn.acrb_clients NOT LIKE '%|vizio|%')
     --        OR onn.acrb_clients <=> '|ALL|')
     --   AND onn.fk_commercial_source_id = 1
     --   AND onn.commercial_client = 'kinetiq'
) AS mt

## Performance Benchmarks

The benchmark section below uses the same 3-day filter for both views:

```sql
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
  AND ts_start < CURRENT_DATE
```

Recommended order:

1. Run the two `EXPLAIN COST` cells first to compare the optimized logical plans
2. Run the existing benchmark cell
3. Run the refined benchmark cell
4. Open the query profile for each run and compare wall-clock time, read bytes, task time, and shuffle-heavy operators

Latest benchmark results:

| Metric | Existing | Refined | Notes |
| --- | --- | --- | --- |
| `COUNT(*)` | 2,756,399,350 | 2,756,400,138 | +788 sessions in refined (no DISTINCT) |
| `COUNT(DISTINCT tvid)` | 13,054,818 | 13,054,818 | identical |
| `COUNT(DISTINCT value)` | 436,760 | 436,760 | identical |

The refined view produces 788 extra sessions (0.00003% of total) because it has no `SELECT DISTINCT`. Unique device and commercial counts are identical, confirming the 788 rows are duplicate sessions for tvids already in the result.

In [0]:
%sql
-- EXPLAIN: Existing view — look for full scan (no partition filter at scan node)
EXPLAIN COST
SELECT COUNT(*), COUNT(DISTINCT tvid), COUNT(DISTINCT value)
FROM dev.mohit_gangwani.test_enhanced_kinetiq_existing
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
AND ts_start < CURRENT_DATE

In [0]:
%sql
-- EXPLAIN: Refined view — look for partition pruning on commercial_client at scan node
EXPLAIN COST
SELECT COUNT(*), COUNT(DISTINCT tvid), COUNT(DISTINCT value)
FROM dev.mohit_gangwani.test_enhanced_kinetiq_refined
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
AND ts_start < CURRENT_DATE

## EXPLAIN COST — Plan Comparison & Findings

EXPLAIN COST plans from Cell 12 (existing) and Cell 13 (refined). Benchmark profiles: existing `69fa4195-7590-435e-b88e-41bd1945ea95`, refined `18cd27a7-c43b-4eab-b402-7251831684f5`.

### What Changed in the Plan

| Area | Existing (Cell 12) | Refined (Cell 13) | Why it matters |
| --- | --- | --- | --- |
| `SELECT DISTINCT` | present on all 29 output columns | removed | drives Shuffle 5.37h + Grouping Aggregate 2.19h in existing; absent in refined |
| Join to hashes table | sort-merge join on `SPLIT_PART(v.tvid, '_', 1) = mt.fk_tvid` to `hashes_production_vizio` | removed — view outputs `mt.tvid` directly | eliminates the join and its associated shuffle entirely |
| Source combination | `UNION` across Vizio and ONN branches | `UNION ALL` across the same branches | removes branch-level deduplication before the scan |
| Projection per CASE column | full `excluded_client_list` / Nielsen / station-vod condition inlined in every prev/next column | `CASE WHEN prev_filter#20933 THEN null` / `CASE WHEN next_filter#20934 THEN null` | single attribute reference replaces repeated string-match per column |
| Scan-level filter | CASE-based `acrb_clients` merged with outer WHERE indirectly | direct pushed filter: `fk_commercial_source_id = 1 AND commercial_client = 'kinetiq' AND NOT (acrb_clients <=> '\|ALL\|') AND (Contains(acrb_clients, '\|kinetiq\|') OR coalesce(acrb_clients, '\|\|') = '\|\|')` | explicit partition pruning co-located with each scan |

### Key Plan Details

**Existing (Cell 12) — plan shape:**

The outer aggregate for the benchmark query sits above a `SELECT DISTINCT` Aggregate node that deduplicates on all 29 output columns, which in turn sits above a `Union false, false`:

```
Aggregate (outer COUNT query)
+- Expand
   +- Aggregate [tvid, zipcode, dma, value, mt_start, ts_start, ts_end, ... 22 more cols]
      (SELECT DISTINCT — 29-column dedup)
      Statistics(sizeInBytes=123.4 TiB, rowCount=2.83E+12)
      +- Union false, false
         Statistics(sizeInBytes=1182.7 TiB, rowCount=2.83E+12)
```

Notably, the planner estimates the same row count (2.83E+12) before and after the Aggregate — this is why the "redundant aggregation" insight was generated. The planner does not have fine-grained statistics on the dedup selectivity, so it assumes DISTINCT removes zero rows. In practice it removed 788 rows, which the planner could not predict.

Each prev/next column still carries the full inline blacklist logic:

```
CASE WHEN (excluded_client_list <=> '|ALL|'
           OR (isnotnull(excluded_client_list)
               AND NOT (excluded_client_list = '||')
               AND NOT Contains(excluded_client_list, '|kinetiq|'))) THEN null
     WHEN prev_nielsen_exclusive THEN null
     WHEN prev_station_vod THEN null
     WHEN true THEN tms_prev_episode_id
END
```

The `commercial_client = 'kinetiq'` and `fk_commercial_source_id = 1` filters are applied via the outer WHERE and are NOT pushed into the scan nodes — the plan must scan more broadly before filtering.

**Refined (Cell 13) — plan shape:**

No `SELECT DISTINCT` Aggregate node is present. Catalyst column-prunes aggressively — since the benchmark query only needs `COUNT(*)`, `COUNT(DISTINCT tvid)`, and `COUNT(DISTINCT value)`, only those two columns are projected from each branch:

```
Aggregate (outer COUNT query)
+- Expand
   +- Union false, false
      :- Project [tvid, external_id AS value]
      :  +- Filter (all conditions pushed to scan)
      :     +- Relation prod.detection.viewing_commercials_dedup_golden
      +- Project [tvid, external_id AS value]
         +- Filter (all conditions pushed to scan)
            +- Relation prod.detection_onn.viewing_commercials_dedup_golden
```

All filters are pushed directly into each scan node:

```
fk_commercial_source_id = 1
AND session_start >= 2026-07-13  AND session_start < 2026-07-16
AND commercial_client = 'kinetiq'
AND NOT (acrb_clients <=> '|ALL|')
AND (Contains(acrb_clients, '|kinetiq|') OR coalesce(acrb_clients, '||') = '||')
```

Predicted post-filter statistics (Databricks partition-aware estimates):

| Branch | Predicted size | Predicted rows |
| --- | --- | --- |
| vizio (`viewing_commercials_dedup_golden`) | 3.2 GiB | 71.6M |
| onn (`viewing_commercials_dedup_golden`) | 146.9 MiB | 3.21M |

**Base table statistics:** `prod.detection.viewing_commercials_dedup_golden` — 3.0 PiB / 2.80E+12 rows raw. `prod.detection_onn.viewing_commercials_dedup_golden` — 25.6 TiB / 2.32E+10 rows.

### Measured Benchmark Results

| Metric | Existing | Refined | Delta |
| --- | --- | --- | --- |
| `COUNT(*)` | 2,756,399,350 | 2,756,400,138 | +788 sessions in refined |
| `COUNT(DISTINCT tvid)` | 13,054,818 | 13,054,818 | identical |
| `COUNT(DISTINCT value)` | 436,760 | 436,760 | identical |
| Duration | 509.0s (8m 29s) | **16.5s** | **−96.8% (30.8× faster)** |
| Read bytes | ~500.0 GiB | ~56.9 GiB | **−88.6%** |
| Task time | 10.04h | 38.9m | **−93.5%** |
| Shuffle | 5.37h | 7.31m | dominant cost in existing was DISTINCT dedup |
| Grouping Aggregate | 2.19h | 18.70m | existing: DISTINCT dedup; refined: benchmark COUNT DISTINCT only |
| Files read | 635 | 331 | **−47.9%** |
| Partitions read | 2 | 2 | same pruning |

> Cache note: existing ran first (0% cache — all remote). Refined ran second (63% cache hit). The 30.8× improvement holds structurally: existing performs ~15× more task work regardless of cache.

### On the 788 Extra Sessions

The refined view has no `SELECT DISTINCT`, so 788 source-level duplicate session rows that the existing view was silently collapsing now appear in the output. `COUNT(DISTINCT tvid)` and `COUNT(DISTINCT value)` are identical in both runs, confirming these rows belong to tvids and commercial placements already in the result set — not new unique devices or new unique commercials. At 0.00003% of 2.756B total sessions this is considered acceptable for a commercial impression report.

In [0]:
%sql
-- Benchmark: Existing view
-- After this runs, open the Query Profile and note: files read, bytes read, duration
SELECT COUNT(*), COUNT(DISTINCT tvid), COUNT(DISTINCT value)
FROM dev.mohit_gangwani.test_enhanced_kinetiq_existing
WHERE ts_start >= CURRENT_DATE - INTERVAL 4 DAY
AND ts_start < CURRENT_DATE  - INTERVAL 1 DAY

In [0]:
%sql
-- Benchmark: Refined view
-- After this runs, open the Query Profile and compare files read, bytes read, duration vs. existing view above
SELECT COUNT(1), COUNT(DISTINCT tvid), COUNT(DISTINCT value)
FROM dev.mohit_gangwani.test_enhanced_kinetiq_refined
WHERE ts_start >= CURRENT_DATE - INTERVAL 4 DAY
AND ts_start < CURRENT_DATE  - INTERVAL 1 DAY

In [0]:
%sql
SELECT COUNT(1), COUNT(DISTINCT tvid), COUNT(DISTINCT value)
FROM prod.datalake_share.enhanced_kinetiq_commercial
WHERE ts_start >= CURRENT_DATE - INTERVAL 4 DAY
AND ts_start < CURRENT_DATE - INTERVAL 1 DAY

In [0]:
%sql
ALTER VIEW cure_deltashare.vizio.enhanced_kinetiq_commercial AS
SELECT
mt.tvid as tvid,
mt.zipcode as zipcode, 
mt.dma as dma, 
mt.external_id as value, 
mt.mt_start as mt_start, 
mt.session_start as ts_start, 
mt.session_end as ts_end, 
CASE WHEN mt.prev_filter THEN NULL
     ELSE mt.tms_prev_episode_id
END as prev_episode_id, 
CASE WHEN mt.prev_filter THEN NULL
     ELSE COALESCE(mt.tms_prev_show_title, mt.tivo_prev_show_title)
END as prev_title, 
mt.prev_ts_start as prev_ts_start, 
mt.prev_ts_end as prev_ts_end, 
CASE WHEN mt.prev_filter THEN NULL
     ELSE COALESCE(mt.tms_prev_channel_callsign, mt.tivo_prev_channel_callsign)
END as prev_channel_callsign,
CASE WHEN mt.prev_filter THEN NULL
     ELSE COALESCE(mt.tms_prev_network_affiliate, mt.tivo_prev_network_affiliate)
END as prev_network_affiliate,
CASE WHEN mt.next_filter THEN NULL 
     ELSE mt.tms_next_episode_id
END as next_episode_id, 
CASE WHEN mt.next_filter THEN NULL
     ELSE COALESCE(mt.tms_next_show_title, mt.tivo_next_show_title)
END as next_title, 
mt.next_ts_start as next_ts_start, 
mt.next_ts_end as next_ts_end, 
CASE WHEN mt.next_filter THEN NULL
     ELSE COALESCE(mt.tms_next_channel_callsign, mt.tivo_next_channel_callsign)
END as next_channel_callsign, 
CASE WHEN mt.next_filter THEN NULL 
     ELSE COALESCE(mt.tms_next_network_affiliate, mt.tivo_next_network_affiliate) 
END as next_network_affiliate,
CASE WHEN mt.prev_filter THEN NULL 
     ELSE mt.live 
END as live, 
mt.brand_name as brand_name, 
mt.title as title, 
mt.duration as duration, 
mt.ip as ip, 
mt.input_category as input_category, 
mt.input_device as input_device, 
CASE WHEN mt.prev_vizio_epg_not_null = False 
     THEN CASE WHEN mt.appb_clients <=> '|ALL|' AND '' <=> mt.app_service THEN mt.app_service 
               WHEN mt.appb_clients IS NOT NULL AND mt.appb_clients != '||' AND mt.appb_clients NOT LIKE '%|vizio|%' THEN 'OBFUSCATED' 
               WHEN mt.appb_clients <=> '|ALL|' THEN 'OBFUSCATED' 
               ELSE mt.app_service END
ELSE mt.app_service 
END as app_service, 
mt.vizio_epg_channel_id as vizio_epg_channel_id, 
mt.vizio_epg_program_id as vizio_epg_program_id,
mt.session_start_hour AS date_partition
FROM (
     SELECT viz.*,
     (COALESCE(viz.excluded_client_list, '||') != '||' AND viz.excluded_client_list NOT LIKE '%|kinetiq|%')
     OR viz.excluded_client_list <=> '|ALL|'
     OR viz.prev_nielsen_exclusive
     OR viz.prev_station_vod AS prev_filter, 
     (COALESCE(viz.excluded_client_list, '||') != '||' AND viz.excluded_client_list NOT LIKE '%|kinetiq|%')
     OR viz.excluded_client_list <=> '|ALL|'
     OR viz.next_nielsen_exclusive
     OR viz.next_station_vod AS next_filter
     FROM prod.detection.viewing_commercials_dedup_golden AS viz
     WHERE viz.fk_commercial_source_id = 1
       AND viz.commercial_client = 'kinetiq'
     UNION ALL
     SELECT onn.*,
     (COALESCE(onn.excluded_client_list, '||') != '||' AND onn.excluded_client_list NOT LIKE '%|kinetiq|%')
     OR onn.excluded_client_list <=> '|ALL|'
     OR onn.prev_nielsen_exclusive
     OR onn.prev_station_vod AS prev_filter, 
     (COALESCE(onn.excluded_client_list, '||') != '||' AND onn.excluded_client_list NOT LIKE '%|kinetiq|%')
     OR onn.excluded_client_list <=> '|ALL|'
     OR onn.next_nielsen_exclusive
     OR onn.next_station_vod AS next_filter
     FROM prod.detection_onn.viewing_commercials_dedup_golden AS onn
     WHERE onn.fk_commercial_source_id = 1
       AND onn.commercial_client = 'kinetiq'
) AS mt
WHERE COALESCE(mt.prev_vizio_epg_not_null, TRUE)
   OR (mt.acrb_clients <=> '|ALL|' AND '' <=> mt.app_service)
   OR COALESCE(mt.acrb_clients, '||') = '||'
   OR mt.acrb_clients LIKE '%|vizio|%'